# Renderer Camera Debug

Use this notebook on the PYNQ-Z1 to load the renderer overlay, find the AXI camera-control block, test register reads/writes, and move the camera in real time.

Before running:

- Copy your `.bit` and matching `.hwh` into the same folder as this notebook, or update `BIT_PATH` below.
- Make sure the HDMI cable is connected to HDMI OUT.
- The block design should contain `ray_renderer_core_axi`, `clk_wiz_0`, and `rgb2dvi_0`.

In [ ]:
from pathlib import Path
import math
import time

from pynq import Overlay, MMIO

BIT_PATH = "renderer.bit"  # Change if your bitstream has another name/path.

bit = Path(BIT_PATH)
if not bit.exists():
    raise FileNotFoundError(f"Could not find {bit.resolve()}. Put the .bit/.hwh here or update BIT_PATH.")

ol = Overlay(str(bit))
ol.download()

def ip_addr(info):
    for key in ("phys_addr", "addr", "base_addr"):
        if key in info:
            return int(info[key])
    return None

def ip_range(info):
    for key in ("addr_range", "range"):
        if key in info:
            return int(info[key])
    return 0x1000

print("Overlay loaded:", bit)
print("IP blocks:")
for name, info in ol.ip_dict.items():
    addr = ip_addr(info)
    if addr is None:
        print(f"  {name:40s} @ <no MMIO address>")
    else:
        print(f"  {name:40s} @ 0x{addr:08X}, range=0x{ip_range(info):X}")

## Find The Camera-Control Base Address

The code below searches for an IP name containing `renderer`, `camera`, or `ray`. If it finds the wrong block, set `CAMERA_IP_NAME` manually to one of the names printed above.

In [ ]:
CAMERA_IP_NAME = None  # Example: "ray_renderer_core_axi_0"

def find_camera_ip(overlay):
    if CAMERA_IP_NAME:
        return CAMERA_IP_NAME, overlay.ip_dict[CAMERA_IP_NAME]

    hints = ("renderer", "camera", "ray")
    matches = []
    for name, info in overlay.ip_dict.items():
        lname = name.lower()
        if ip_addr(info) is not None and any(h in lname for h in hints):
            matches.append((name, info))

    if not matches:
        raise RuntimeError("Could not auto-find renderer IP. Set CAMERA_IP_NAME manually.")
    if len(matches) > 1:
        print("Multiple possible camera IPs found; using the first:")
        for name, info in matches:
            print(f"  {name:40s} @ 0x{ip_addr(info):08X}")
    return matches[0]

cam_ip_name, cam_ip_info = find_camera_ip(ol)
CAMERA_BASE = ip_addr(cam_ip_info)
if CAMERA_BASE is None:
    raise RuntimeError(f"{cam_ip_name} has no MMIO base address. Check the .hwh file and AXI address assignment.")
CAMERA_RANGE = ip_range(cam_ip_info)
mmio = MMIO(CAMERA_BASE, CAMERA_RANGE)

print("Using camera IP:", cam_ip_name)
print(f"Base address: 0x{CAMERA_BASE:08X}")

## Register Map Helpers

In [ ]:
REG_CONTROL = 0x00
REG_STATUS = 0x04

REG_OX = 0x10
REG_OY = 0x14
REG_OZ = 0x18

REG_FWD_X = 0x20
REG_FWD_Y = 0x24
REG_FWD_Z = 0x28

REG_RIGHT_X = 0x30
REG_RIGHT_Y = 0x34
REG_RIGHT_Z = 0x38

REG_UP_X = 0x40
REG_UP_Y = 0x44
REG_UP_Z = 0x48

CAMERA_REGS = [
    ("Ox", REG_OX), ("Oy", REG_OY), ("Oz", REG_OZ),
    ("fwd_x", REG_FWD_X), ("fwd_y", REG_FWD_Y), ("fwd_z", REG_FWD_Z),
    ("right_x", REG_RIGHT_X), ("right_y", REG_RIGHT_Y), ("right_z", REG_RIGHT_Z),
    ("up_x", REG_UP_X), ("up_y", REG_UP_Y), ("up_z", REG_UP_Z),
]

def q2_13(value):
    raw = int(round(value * 8192.0))
    raw = max(-32768, min(32767, raw))
    return raw & 0xFFFF

def from_q2_13(raw):
    raw &= 0xFFFF
    if raw & 0x8000:
        raw -= 0x10000
    return raw / 8192.0

def norm(v):
    length = math.sqrt(sum(c * c for c in v))
    if length == 0:
        raise ValueError("zero-length vector")
    return tuple(c / length for c in v)

def cross(a, b):
    return (
        a[1] * b[2] - a[2] * b[1],
        a[2] * b[0] - a[0] * b[2],
        a[0] * b[1] - a[1] * b[0],
    )

def read_camera():
    rows = []
    for name, offset in CAMERA_REGS:
        raw = mmio.read(offset)
        rows.append((name, raw & 0xFFFF, from_q2_13(raw)))
    return rows

def print_camera():
    print(f"status pending: {mmio.read(REG_STATUS) & 1}")
    for name, raw, value in read_camera():
        print(f"{name:8s} raw=0x{raw:04X} value={value: .5f}")

def wait_committed(timeout_s=0.2):
    deadline = time.monotonic() + timeout_s
    while mmio.read(REG_STATUS) & 1:
        if time.monotonic() > deadline:
            raise TimeoutError("camera commit did not complete")
        time.sleep(0.001)

def write_basis(position, fwd, right, up, wait=True):
    values = (
        (REG_OX, position[0]), (REG_OY, position[1]), (REG_OZ, position[2]),
        (REG_FWD_X, fwd[0]), (REG_FWD_Y, fwd[1]), (REG_FWD_Z, fwd[2]),
        (REG_RIGHT_X, right[0]), (REG_RIGHT_Y, right[1]), (REG_RIGHT_Z, right[2]),
        (REG_UP_X, up[0]), (REG_UP_Y, up[1]), (REG_UP_Z, up[2]),
    )
    for offset, value in values:
        mmio.write(offset, q2_13(value))
    mmio.write(REG_CONTROL, 1)
    if wait:
        wait_committed()

def write_yaw_pitch(position=(-0.35, -0.35, 0.45), yaw_deg=45.0, pitch_deg=-45.0, wait=True):
    yaw = math.radians(yaw_deg)
    pitch = math.radians(pitch_deg)
    cp = math.cos(pitch)
    fwd = norm((cp * math.cos(yaw), cp * math.sin(yaw), math.sin(pitch)))
    world_up = (0.0, 0.0, 1.0)
    right = norm(cross(fwd, world_up))
    up = cross(right, fwd)
    write_basis(position, fwd, right, up, wait=wait)

def write_look_at(position=(-0.35, -0.35, 0.45), target=(0.0, 0.0, 0.0), wait=True):
    fwd = norm(tuple(t - p for p, t in zip(position, target)))
    world_up = (0.0, 0.0, 1.0)
    right = norm(cross(fwd, world_up))
    up = cross(right, fwd)
    write_basis(position, fwd, right, up, wait=wait)

print_camera()

## Quick Register Sanity Test

This writes a known camera pose, commits it, and reads the shadow registers back. If the values read back correctly, AXI-Lite is alive.

In [ ]:
write_yaw_pitch(position=(-0.35, -0.35, 0.45), yaw_deg=45, pitch_deg=-45)
print_camera()

## Manual Camera Moves

`yaw_deg` is in the XY plane. `pitch_deg=-45` looks downward at 45 degrees.

In [ ]:
# Try a few views. Re-run/edit this cell while watching HDMI output.
write_yaw_pitch(position=(-0.35, -0.35, 0.45), yaw_deg=45, pitch_deg=-45)
# write_yaw_pitch(position=(-0.45, -0.10, 0.45), yaw_deg=20, pitch_deg=-45)
# write_yaw_pitch(position=(-0.10, -0.45, 0.45), yaw_deg=70, pitch_deg=-45)
# write_look_at(position=(-0.55, -0.55, 0.65), target=(0.0, 0.0, 0.1))

## Optional Interactive Controls

If `ipywidgets` is available on your PYNQ image, this gives you sliders. Move them slowly; every slider update writes AXI registers and commits on the next video frame.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    x = widgets.FloatSlider(value=-0.35, min=-0.9, max=0.9, step=0.01, description="x")
    y = widgets.FloatSlider(value=-0.35, min=-0.9, max=0.9, step=0.01, description="y")
    z = widgets.FloatSlider(value=0.45, min=0.05, max=1.5, step=0.01, description="z")
    yaw = widgets.FloatSlider(value=45.0, min=-180.0, max=180.0, step=1.0, description="yaw")
    pitch = widgets.FloatSlider(value=-45.0, min=-85.0, max=-5.0, step=1.0, description="pitch")

    def update(change=None):
        write_yaw_pitch(position=(x.value, y.value, z.value), yaw_deg=yaw.value, pitch_deg=pitch.value, wait=False)

    for slider in (x, y, z, yaw, pitch):
        slider.observe(update, names="value")

    display(widgets.VBox([x, y, z, yaw, pitch]))
    update()
except ImportError:
    print("ipywidgets is not installed. Use the manual cells above.")